In [2]:
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

In [3]:
load_dotenv()

docs = [
    Document(page_content="The sky is blue because of Rayleigh scattering.", metadata={"source": "doc1"}),
    Document(page_content="The capital of France is Paris.", metadata={"source": "doc2"}),
    Document(page_content="Water boils at 100 degrees Celsius at sea level.", metadata={"source": "doc3"}),
    Document(page_content="Photosynthesis is the process plants use to make food.", metadata={"source": "doc4"})
]

Create embeddings and vector store

In [4]:
embeddings = OpenAIEmbeddings()

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory=None
)

retriever = vectorstore.as_retriever(search_kwargs={'k': 2})

print('Sample vector store ready.')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Sample vector store ready.


Step 2: Test Retrieval Without Transformation

In [6]:
raw_query = 'Why is the sky blue and what temperature does water boil?'

# Retrieve without transformation
raw_docs = retriever.invoke(raw_query)

print('Without transformation:')

for doc in raw_docs:
    print(f'- {doc.page_content}')

Without transformation:
- Water boils at 100 degrees Celsius at sea level.
- The sky is blue because of Rayleigh scattering.


Step 3: Define a Query Transformer

In [9]:
# LLM for query rewriting
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Prompt for query transformation
transform_prompt = ChatPromptTemplate.from_messages([
    ("system", "Rewrite the user's question to make it more specific and likely to retrieve relevant information from a knowledge base."),
    ("human", "{question}")
])

# Chain to transform the query
query_transformer = transform_prompt | llm | StrOutputParser()

Step 4: Test Retrieval With Transformation

In [12]:
# Transform the query
transformed_query = query_transformer.invoke({'question': raw_query})
print(f'Transformed query: {transformed_query}')

# Retrieve with transformed query
transformed_docs = retriever.invoke(transformed_query)

print('\nWith transformation:')
for doc in transformed_docs:
    print(f'- {doc.page_content}')

Transformed query: What causes the blue color of the sky, and at what temperature does water boil at standard atmospheric pressure?

With transformation:
- Water boils at 100 degrees Celsius at sea level.
- The sky is blue because of Rayleigh scattering.


In [11]:
# Transform the query
transformed_query1 = query_transformer.invoke({'question': raw_query})
print(f'Transformed query: {transformed_query1}')


Transformed query: What causes the blue color of the sky, and at what temperature does water boil at sea level?
